# 01 Data Contract

Этот ноутбук проверяет входные данные перед запуском моделей:

1. **Загрузка genotype** — читаем `.rds` файл через `pyreadr`
2. **Загрузка phenotype** — читаем `.tsv` файл (id, pop, trait)
3. **Проверка пересечения id** — убеждаемся, что образцы есть и в genotype, и в phenotype
4. **Проверка пропусков** — смотрим, сколько `NA` в фенотипе (это те, которые нужно предсказать)

**Для кого**: для тех, кто хочет понять, какие данные пришли и всё ли в порядке перед обучением.

In [ ]:
# Настройка: добавим путь к gp_py в sys.path
import sys
sys.path.insert(0, '..')

from pathlib import Path
import pandas as pd
import numpy as np

# Загружаем модули gp_py
from gp_py.io import fn_load_genotype, fn_load_phenotype
from gp_py.schema import MergedData

## 1. Загрузка genotype

In [ ]:
# Путь к тестовым данным
DATA_DIR = Path("../inst/exec_Rscript/input")
FNAME_GENO = DATA_DIR / "test_geno.Rds"
FNAME_PHENO = DATA_DIR / "test_pheno.tsv"

print("Загружаем genotype...")
G = fn_load_genotype(str(FNAME_GENO), verbose=True)
print(f"  Размер genotype матрицы: {G.shape[0]} samples x {G.shape[1]} markers")
print(f"  Первые 5 колонок: {list(G.columns[:5])}")

In [ ]:
G.tail()

## 2. Загрузка phenotype

In [ ]:
print("Загружаем phenotype...")
list_pheno = fn_load_phenotype(
    str(FNAME_PHENO),
    sep="\t",
    header=True,
    idx_col_id=1,
    idx_col_pop=2,
    idx_col_y=3,
    verbose=True
)

df_pheno = list_pheno["df"]
print(f"  Число строк: {len(df_pheno)}")
print(f"  Колонки: {list(df_pheno.columns)}")
print(f"  ID колонка: {list_pheno['id_col']}")
print(f"  POP колонка: {list_pheno['pop_col']}")
print(f"  Trait колонка: {list_pheno['y_col']}")

## 3. Пересечение ID (join)

In [ ]:
# Получаем списки ID из genotype и phenotype
geno_ids = set(G.index.astype(str)) if G.index.name else set()
pheno_ids = set(df_pheno[list_pheno['id_col']].astype(str))

common_ids = geno_ids & pheno_ids
only_geno = geno_ids - pheno_ids
only_pheno = pheno_ids - geno_ids

print("=== Пересечение ID ===")
print(f"  ID в genotype: {len(geno_ids)}")
print(f"  ID в phenotype: {len(pheno_ids)}")
print(f"  Общие ID (будут использованы): {len(common_ids)}")
print(f"  Только в genotype (потеряются): {len(only_geno)}")
print(f"  Только в phenotype (незачем предсказывать): {len(only_pheno)}")

## 4. Пропуски в phenotype (что будем предсказывать)

In [ ]:
y_col = list_pheno['y_col']
y_series = df_pheno[y_col]

known_mask = y_series.notna()
missing_mask = y_series.isna()

print("=== Пропуски в trait ===")
print(f"  Всего записей: {len(y_series)}")
print(f"  Известные фенотипы (для обучения): {known_mask.sum()}")
print(f"  Пропущенные фенотипы (для предсказания): {missing_mask.sum()}")

if missing_mask.sum() > 0:
    missing_ids = df_pheno.loc[missing_mask, list_pheno['id_col']].head(5).tolist()
    print(f"  Пример ID с пропусками: {missing_ids} ...")

## 5. Population distribution

In [ ]:
pop_col = list_pheno['pop_col']
pop_counts = df_pheno[pop_col].value_counts()
print("=== Population distribution ===")
print(pop_counts.to_string())

## Итог

- Мы загрузили genotype (500 × 1000) и phenotype (500 записей)
- Общих ID: столько, сколько войдёт в анализ
- Пропуски в trait: столько значений нужно предсказать после обучения моделей
- Population: несколько групп для кросс-валидации

**Следующий шаг**: перейти к ноутбуку `02_cv_design`, чтобы увидеть, как данные разбиваются на fold-ы.